<a href="https://colab.research.google.com/github/drod75/nyc_taxi_analysis/blob/main/notebooks/data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering

The point of this notebook is to use our data we collected to create new data. In particular we will take our taxi trip data and turn it into trip metric data, showcasing metrics for each unique trip within the data, and unique statistics per each day of the summer.

## Setup

In [1]:
# importing libraries, pandas for data manipulation, requests for data gathering, colab for environment variables
import pandas as pd
import geopandas
from google.colab import userdata
from google.colab import drive
from pprint import pprint as print

In [2]:
# mounting drive for future use
drive.mount('drive')

Mounted at drive


In [4]:
# importing datasets from Google Drive
taxi_df = pd.read_csv("/content/drive/MyDrive/taxi_summer_data.csv")
zone_df = geopandas.read_file("/content/drive/MyDrive/taxi_zones_data.json")

In [9]:
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707066 entries, 0 to 11707065
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   tpep_pickup_datetime   object 
 1   tpep_dropoff_datetime  object 
 2   passenger_count        float64
 3   trip_distance          float64
 4   ratecodeid             float64
 5   pulocationid           int64  
 6   dolocationid           int64  
 7   payment_type           int64  
 8   fare_amount            float64
 9   extra                  float64
 10  mta_tax                float64
 11  tip_amount             float64
 12  tolls_amount           float64
 13  improvement_surcharge  float64
 14  total_amount           float64
 15  congestion_surcharge   float64
 16  airport_fee            float64
 17  cbd_congestion_fee     float64
dtypes: float64(13), int64(3), object(2)
memory usage: 1.6+ GB


In [10]:
zone_df.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 263 entries, 0 to 262
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   shape_leng  263 non-null    float64 
 1   shape_area  263 non-null    float64 
 2   zone        263 non-null    object  
 3   locationid  263 non-null    int32   
 4   borough     263 non-null    object  
 5   geometry    263 non-null    geometry
dtypes: float64(2), geometry(1), int32(1), object(2)
memory usage: 11.4+ KB


## Individual Trips Statistics

This branch of our summer data will condense every unique trip and gather there aggregrated statistics, such as fare data, trip length data, if it was in the borough, etc.

In [5]:
# displaying information about the taxi dataframe
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11707066 entries, 0 to 11707065
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   tpep_pickup_datetime   object 
 1   tpep_dropoff_datetime  object 
 2   passenger_count        float64
 3   trip_distance          float64
 4   ratecodeid             float64
 5   pulocationid           int64  
 6   dolocationid           int64  
 7   payment_type           int64  
 8   fare_amount            float64
 9   extra                  float64
 10  mta_tax                float64
 11  tip_amount             float64
 12  tolls_amount           float64
 13  improvement_surcharge  float64
 14  total_amount           float64
 15  congestion_surcharge   float64
 16  airport_fee            float64
 17  cbd_congestion_fee     float64
dtypes: float64(13), int64(3), object(2)
memory usage: 1.6+ GB


In [6]:
# grouping taxi data by pickup and dropoff locations to calculate metrics
trip_df = taxi_df.groupby(['pulocationid', 'dolocationid'])['dolocationid'].count()
trip_df = trip_df.reset_index(name='trip_count')
trip_df.head()

,pulocationid,dolocationid,trip_count
0,1,1,1553
1,1,23,1
2,1,25,1
3,1,40,1
4,1,48,4


In [7]:
# sorting trip data by pickup and dropoff location IDs
trip_df.sort_values(by=['pulocationid','dolocationid'], ascending=True, inplace=True)
trip_df.head(10)

,pulocationid,dolocationid,trip_count
0,1,1,1553
1,1,23,1
2,1,25,1
3,1,40,1
4,1,48,4
5,1,50,2
6,1,68,1
7,1,79,1
8,1,107,1
9,1,113,1


In [11]:
# merging trip data with zone borough data based on pickup location ID
trip_df = pd.merge(trip_df, zone_df, left_on='pulocationid', right_on='locationid')
trip_df.info()
trip_df.drop(columns=['geometry', 'shape_leng', 'shape_area', 'locationid','trip_count'], inplace=True)
trip_df.rename(columns={"zone": "puzone", "borough": "puborough"}, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44902 entries, 0 to 44901
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   pulocationid  44902 non-null  int64   
 1   dolocationid  44902 non-null  int64   
 2   trip_count    44902 non-null  int64   
 3   shape_leng    44902 non-null  float64 
 4   shape_area    44902 non-null  float64 
 5   zone          44902 non-null  object  
 6   locationid    44902 non-null  int32   
 7   borough       44902 non-null  object  
 8   geometry      44902 non-null  geometry
dtypes: float64(2), geometry(1), int32(1), int64(3), object(2)
memory usage: 2.9+ MB


In [12]:
# merging trip data with zone data based on dropoff location ID
trip_df = pd.merge(trip_df, zone_df, left_on='dolocationid', right_on='locationid')
trip_df.drop(columns=['geometry', 'shape_leng', 'shape_area', 'locationid'], inplace=True)
trip_df.rename(columns={"zone": "dozone", "borough": "doborough"}, inplace=True)
trip_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44604 entries, 0 to 44603
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   pulocationid  44604 non-null  int64 
 1   dolocationid  44604 non-null  int64 
 2   puzone        44604 non-null  object
 3   puborough     44604 non-null  object
 4   dozone        44604 non-null  object
 5   doborough     44604 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.0+ MB


In [13]:
# checking if the pickup and dropoff boroughs are the same
trip_df['interborough'] = (trip_df['puborough'] == trip_df['doborough'])

In [14]:
# displaying the first few rows of the merged trip dataframe
trip_df.head()

,pulocationid,dolocationid,puzone,puborough,dozone,doborough,interborough
0,1,1,Newark Airport,EWR,Newark Airport,EWR,True
1,1,23,Newark Airport,EWR,Bloomfield/Emerson Hill,Staten Island,False
2,1,25,Newark Airport,EWR,Boerum Hill,Brooklyn,False
3,1,40,Newark Airport,EWR,Carroll Gardens,Brooklyn,False
4,1,48,Newark Airport,EWR,Clinton East,Manhattan,False


## Export

In [15]:
# exporting datasets to google drive
trip_df.to_csv('/content/drive/MyDrive/trip_data.csv', index=False)